# KL Expressibility on IQM Spark (Hardware–Hardware Overlaps)

**Authors:** Koło Naukowe Axion

This notebook estimates **KL expressibility** on the IQM Spark quantum computer by comparing the empirical distribution of **hardware–hardware overlaps**

$$F = \mathrm{Tr}(\rho_a \rho_b)$$

against the analytic **Haar fidelity distribution** via

$$D_{\mathrm{KL}}(P_{\mathrm{hardware}}(F)\,\|\,P_{\mathrm{Haar}}(F)).$$

## Protocol

1. For a fixed ansatz $U(\theta)$ and depth, draw two independent parameter vectors $\theta_a, \theta_b \sim U[0, 2\pi)$.
2. Prepare both circuits on IQM Spark and run full $3^n$ Pauli-basis **quantum state tomography** for each state.
3. Reconstruct $\rho_a$, $\rho_b$ from measurement counts (linear inversion + PSD projection).
4. Compute $F_{\mathrm{linear}}$ and $F_{\mathrm{physical}}$; bin samples and compute $D_{\mathrm{KL}}$ vs discretized Haar.

Tomography methodology matches [`full_odra_fidelity.ipynb`](full_odra_fidelity.ipynb).

## Runtime budget

For $n=5$ qubits each fidelity sample requires **$2 \times 3^5 = 486$** tomography circuits. On IQM Spark, 243 circuits @ 1024 shots takes about **2 minutes** per state, so each pair is about **4 minutes**. With a **13-hour** budget, defaults use **30 pairs per (ansatz, depth)** across 2 ansatze and depths $\{2,4,6\}$.

## Hyperparameter pilot and full study

Before the full sweep, run the KL pilot to choose `SHOTS`, `N_SAMPLES`, and `N_BINS`.
Defaults use depths $\{2,4,6\}$, shot grid up to **8192**, sample grid up to **30** pairs, and 3 pilot pairs per shot setting (~**42 h** total QPU budget including sample and iteration stages).

**Full study (pilot + production + offline QPU/Sim/Haar analysis):**

```bash
export IQM_TOKEN="..."
./scripts/run_iqm_kl_full_study.sh
```

Pilot only:

```bash
python scripts/run_iqm_kl_pilot.py --pilot-id kl_pilot_paper
```

See [`iqm_kl_pilot_methodology.md`](iqm_kl_pilot_methodology.md). Apply the resulting `kl_protocol_recommendation.json` via `scripts/run_iqm_kl_expressibility.py --protocol-json ...` or copy the chosen values into the configuration cell below.

**Production hardware protocol (fixed budget, no half-width pilot):** see [`iqm_kl_hardware_methodology.html`](iqm_kl_hardware_methodology.html). Run `bash scripts/run_iqm_kl_hardware_study.sh` and analyze with `python scripts/analyze_iqm_kl_hardware.py --run-dir ...`.


## 1. Imports & Configuration

In [ ]:
from qbanknote.paths import ensure_importable
ensure_importable()

import csv
import getpass
import json
import math
import os
import time
from datetime import datetime, timezone
from itertools import product
from pathlib import Path
from typing import Callable

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from qiskit import QuantumCircuit, transpile
from qiskit.circuit import ParameterVector
from qiskit.quantum_info import Statevector

NUM_QUBITS = 5
DEPTHS = [2, 4, 6]
ANSATZES = {
    "ansatz_odra": None,
    "ansatz_simulator": None,
}
SEED = 42
N_SAMPLES = 30
SHOTS = 1024
MINUTES_PER_STATE_TOMOGRAPHY = 2.0
N_BINS = 150
EPS = 1e-12
DIM = 2 ** NUM_QUBITS
IQM_URL = os.environ.get("IQM_URL", "https://odra5.e-science.pl/").strip()
OPTIMIZATION_LEVEL = 1
MAX_CIRCUITS_PER_JOB = 250
NOTEBOOK_DIR = Path(".").resolve()

circuits_per_pair = 2 * (3 ** NUM_QUBITS)
total_circuits = len(ANSATZES) * len(DEPTHS) * N_SAMPLES * circuits_per_pair
est_minutes = len(ANSATZES) * len(DEPTHS) * N_SAMPLES * 2.0 * MINUTES_PER_STATE_TOMOGRAPHY
print(f"n={NUM_QUBITS}, circuits per fidelity pair: {circuits_per_pair}")
print(
    f"Planned run: {len(ANSATZES)} ansatze x {len(DEPTHS)} depths x "
    f"{N_SAMPLES} pairs = {total_circuits} tomography circuits x {SHOTS} shots"
)
print(f"Estimated wall time: {est_minutes:.0f} min ({est_minutes / 60:.1f} h)")


## 2. Ansatz Definitions

In [ ]:
from qbanknote.ansatzes import (
    ansatz_trimmed_reverse_q0_param_count,
    odra_ansatz as ansatz_odra,
    simulator_ansatz as ansatz_simulator,
)

ANSATZES["ansatz_odra"] = ansatz_odra
ANSATZES["ansatz_simulator"] = ansatz_simulator

for depth in DEPTHS:
    n_params = ansatz_trimmed_reverse_q0_param_count(NUM_QUBITS, depth)
    print(f"depth={depth}: n_params={n_params}")


## 3. Tomography, KL, and Hardware Helpers

In [ ]:
from qbanknote.iqm import connect_to_iqm_backend, transpile_for_backend, normalize_counts
from qbanknote.tomography import (
    all_basis_settings,
    add_tomography_rotations,
    expectation_from_counts,
    reconstruct_rho,
    project_to_physical,
    hardware_overlap,
    run_tomography_jobs,
    tomography_density_matrices,
)
from qbanknote.metrics import (
    haar_pdf_fidelity,
    binned_distributions,
    kl_divergence,
    bind_ansatz,
    sample_hardware_fidelities,
    compute_kl_for_fidelities,
    circuits_per_fidelity_sample,
    total_expressibility_circuits,
    estimate_wall_time_minutes,
    run_kl_self_check as run_self_check,
)


## 4. Self-check (no hardware)

In [ ]:
run_self_check()

## 5. Connect to IQM Spark

Set `IQM_TOKEN` in your environment or enter the token when prompted.


In [ ]:
iqm_backend = connect_to_iqm_backend(IQM_URL)
print(f"Connected to backend: {iqm_backend}  (n_qubits = {iqm_backend.num_qubits})")


## 6. Hardware sweep

In [ ]:
stamp = datetime.now(tz=timezone.utc).strftime("%Y%m%d_%H%M%S")
output_dir = NOTEBOOK_DIR / f"iqm_kl_expressibility_{stamp}"
output_dir.mkdir(parents=True, exist_ok=True)

summary_rows = []
fidelity_rows = []
t_start = time.perf_counter()

for depth in DEPTHS:
    for ansatz_name, ansatz_fn in ANSATZES.items():
        depth_seed = SEED + 100 * depth + (1 if ansatz_name == "ansatz_simulator" else 0)
        print(f"\n=== {ansatz_name} depth={depth} (seed={depth_seed}) ===")

        sample_rows = sample_hardware_fidelities(
            iqm_backend,
            ansatz_fn,
            n_qubits=NUM_QUBITS,
            depth=depth,
            n_samples=N_SAMPLES,
            seed=depth_seed,
            shots=SHOTS,
            optimization_level=OPTIMIZATION_LEVEL,
            seed_transpiler=None,
            max_circuits_per_job=MAX_CIRCUITS_PER_JOB,
            ansatz_label=ansatz_name,
        )

        f_phys = np.array([r["fidelity_physical"] for r in sample_rows])
        f_lin = np.array([r["fidelity_linear"] for r in sample_rows])
        kl_phys, _, _, _ = compute_kl_for_fidelities(f_phys, DIM, N_BINS, EPS)
        kl_lin, _, _, _ = compute_kl_for_fidelities(f_lin, DIM, N_BINS, EPS)

        summary_rows.append(
            {
                "ansatz": ansatz_name,
                "depth": depth,
                "n_qubits": NUM_QUBITS,
                "n_samples": N_SAMPLES,
                "shots": SHOTS,
                "n_bins": N_BINS,
                "eps": EPS,
                "seed": depth_seed,
                "kl_physical": kl_phys,
                "kl_linear": kl_lin,
                "f_physical_mean": float(np.mean(f_phys)),
                "f_physical_std": float(np.std(f_phys)),
                "f_linear_mean": float(np.mean(f_lin)),
                "f_linear_std": float(np.std(f_lin)),
            }
        )
        for row in sample_rows:
            fidelity_rows.append({"ansatz": ansatz_name, "depth": depth, **row})

wall_min = (time.perf_counter() - t_start) / 60.0

results_df = pd.DataFrame(summary_rows)
fidelities_df = pd.DataFrame(fidelity_rows)
results_path = output_dir / "iqm_kl_results.csv"
fidelities_path = output_dir / "iqm_kl_fidelities.csv"
manifest_path = output_dir / "run_manifest.json"
results_df.to_csv(results_path, index=False)
fidelities_df.to_csv(fidelities_path, index=False)

manifest = {
    "created_utc": datetime.now(tz=timezone.utc).isoformat(),
    "backend": str(iqm_backend),
    "iqm_url": IQM_URL,
    "source_notebook": "evaluation_and_comparison/iqm_kl_expressibility.ipynb",
    "tomography_source": "full_odra_fidelity.ipynb",
    "method": "hardware_hardware_overlap_tomography",
    "fidelity_definition": "Tr(rho_a @ rho_b) from full 3^n Pauli tomography",
    "kl_direction": "P_hardware || P_Haar",
    "n_qubits": NUM_QUBITS,
    "dim": DIM,
    "depths": list(DEPTHS),
    "ansatzes": list(ANSATZES.keys()),
    "n_samples": N_SAMPLES,
    "shots": SHOTS,
    "n_bins": N_BINS,
    "eps": EPS,
    "seed": SEED,
    "optimization_level": OPTIMIZATION_LEVEL,
    "max_circuits_per_job": MAX_CIRCUITS_PER_JOB,
    "circuits_per_fidelity_sample": circuits_per_pair,
    "total_tomography_circuits": total_circuits,
    "wall_time_minutes": wall_min,
    "outputs": [results_path.name, fidelities_path.name],
}
manifest_path.write_text(json.dumps(manifest, indent=2, sort_keys=True) + "\n")
print(f"\nSaved outputs to {output_dir}")
print(f"Total wall time: {wall_min:.1f} min")
results_df


## 7. Comparison table

In [ ]:
print("KL comparison (lower is better, physical projection):")
print("depth | ansatz           | KL_physical | F_phys_mean")
print("-" * 55)
for _, row in results_df.iterrows():
    print(
        f"{int(row['depth']):>5} | {row['ansatz']:<16} | "
        f"{row['kl_physical']:.6f}    | {row['f_physical_mean']:.4f}"
    )


## 8. Plots

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
for ansatz_name in ANSATZES:
    sub = results_df[results_df["ansatz"] == ansatz_name].sort_values("depth")
    ax.plot(sub["depth"], sub["kl_physical"], marker="o", label=ansatz_name)
ax.set_xlabel("Depth")
ax.set_ylabel("KL(P_hardware || P_Haar)")
ax.set_title("KL vs depth on IQM Spark (lower is better)")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


In [ ]:
PLOT_DEPTH = DEPTHS[-1]
fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharey=True)

for ax, ansatz_name in zip(axes, ANSATZES):
    sub = fidelities_df[
        (fidelities_df["ansatz"] == ansatz_name) & (fidelities_df["depth"] == PLOT_DEPTH)
    ]
    f_vals = sub["fidelity_physical"].to_numpy()
    _, mids, p_emp, p_haar = binned_distributions(f_vals, DIM, N_BINS)
    width = 1.0 / N_BINS
    ax.bar(mids, p_emp / width, width=width, alpha=0.5, label="hardware empirical")
    haar_curve = haar_pdf_fidelity(mids, DIM)
    ax.plot(mids, haar_curve, color="black", lw=2, label="Haar density")
    ax.set_title(f"{ansatz_name} depth={PLOT_DEPTH}")
    ax.set_xlabel("F = Tr(rho_a rho_b)")
    ax.set_ylabel("density")
    ax.legend()

plt.suptitle("Hardware overlap histogram vs Haar (physical projection)")
plt.tight_layout()
plt.show()
